In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import tensorflow as tf
tf.__version__

In [1]:
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

2026-03-11 19:08:14.095339: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773256094.282693      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773256094.344319      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773256094.777473      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773256094.777514      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773256094.777517      55 computation_placer.cc:177] computation placer alr

In [2]:
base_path = "/kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images"

In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# base_path = "/kaggle/input/datasets/iarunava/cell-images-for-detecting-malaria/cell_images/cell_images"

# Training — augmentation + rescale
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=180,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest"
)
train_generator = train_datagen.flow_from_directory(
    base_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=42
)

# Validation — rescale only, NO augmentation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)
val_generator = val_datagen.flow_from_directory(
    base_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print(train_generator.class_indices)

Found 22048 images belonging to 2 classes.
Found 5510 images belonging to 2 classes.
{'Parasitized': 0, 'Uninfected': 1}


In [4]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger

callbacks = [
    ModelCheckpoint("vgg19_malaria.keras", monitor="val_accuracy",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=7,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
    CSVLogger("vgg19_malaria_log.csv")
]

In [5]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Sequential

In [6]:
base_model = VGG19(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

# Freeze convolution layers
for layer in base_model.layers:
    layer.trainable = False

# Custom classifier
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)

outputs = Dense(train_generator.num_classes, activation='softmax')(x)

vgg_model = Model(inputs=base_model.input, outputs=outputs)

I0000 00:00:1773256130.521869      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [7]:
vgg_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv4 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 20,288,066 (77.39 MB)

 Trainable params: 263,682 (1.01 MB)

 Non-trainable params: 20,024,384 (76.39 MB)

In [8]:
from tensorflow.keras.optimizers import Adam

vgg_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = vgg_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=callbacks
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


I0000 00:00:1773256155.440513     136 service.cc:152] XLA service 0x7cbff000f220 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773256155.440550     136 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773256155.986602     136 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/689 ━━━━━━━━━━━━━━━━━━━━ 1:15:06 7s/step - accuracy: 0.5312 - loss: 0.8654

I0000 00:00:1773256160.435641     136 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 340ms/step - accuracy: 0.6571 - loss: 0.6130
Epoch 1: val_accuracy improved from -inf to 0.78312, saving model to vgg19_malaria.keras
689/689 ━━━━━━━━━━━━━━━━━━━━ 282s 401ms/step - accuracy: 0.6572 - loss: 0.6129 - val_accuracy: 0.7831 - val_loss: 0.4461 - learning_rate: 1.0000e-04
Epoch 2/15
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.8533 - loss: 0.3630
Epoch 2: val_accuracy improved from 0.78312 to 0.81089, saving model to vgg19_malaria.keras
689/689 ━━━━━━━━━━━━━━━━━━━━ 126s 183ms/step - accuracy: 0.8533 - loss: 0.3630 - val_accuracy: 0.8109 - val_loss: 0.3924 - learning_rate: 1.0000e-04
Epoch 3/15
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.8784 - loss: 0.3069
Epoch 3: val_accuracy did not improve from 0.81089
689/689 ━━━━━━━━━━━━━━━━━━━━ 130s 189ms/step - accuracy: 0.8784 - loss: 0.3069 - val_accuracy: 0.7855 - val_loss: 0.4426 - learning_rate: 1.0000e-04
Epoch 4/15
689/689 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - accuracy: 0.

In [ ]:
vgg_model.save("vgg19_malaria.keras")

In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Make sure generator is not shuffled
val_generator.reset()

# Predict on validation set
pred_probs = model.predict(val_generator, verbose=1)

# Convert probabilities to 0 or 1
pred_labels = (pred_probs > 0.5).astype(int).reshape(-1)

# True labels
true_labels = val_generator.classes

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)
print("Confusion Matrix:")
print(cm)

In [ ]:
import seaborn as sns

plt.figure(figsize=(5,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Parasitized','Uninfected'],
            yticklabels=['Parasitized','Uninfected'])

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
print(classification_report(true_labels, pred_labels, target_names=['Parasitized','Uninfected']))